# Astronomical Data File Formats
## A practical guide using Gaia DR2 data with astropy

**Data:** Gaia Data Release 2 (Gaia DR2)  
**Subsample:** Stars with $16 \leq G < 17$ mag (~78.9 million stars, 13.4 GB)  
**Paper:** Gaia Collaboration, Brown et al. (2018), A&A 616, A1  
**arXiv:** [1804.09365](https://arxiv.org/abs/1804.09365)  

**Key tool:** `astropy` — [https://docs.astropy.org/en/stable/](https://docs.astropy.org/en/stable/)  
**astropy I/O reference:** [https://docs.astropy.org/en/stable/io/unified.html](https://docs.astropy.org/en/stable/io/unified.html)

---

## Learning objectives

After this tutorial you will be able to:
1. Understand the structure and purpose of **FITS**, **VOTable**, **ECSV**, **HDF5**, and **CSV** file formats.
2. Read and write astronomical catalogs using the **`astropy.table.Table`** interface.
3. Manipulate data: **filter**, **sort**, **add columns**, compute derived quantities.
4. Preserve **metadata** (units, descriptions, provenance) through format conversions.
5. Choose the right format for your science case.

---

## 1. Why do different file formats exist?

Astronomical datasets come in many sizes and use cases:
- A **79-million-star Gaia catalog** needs binary storage (13 GB as FITS, >50 GB as plain text)
- A **results table** of 20 measured galaxy properties needs human-readable format + units
- A **pipeline output** needs self-describing metadata and fast random access
- A **webpage table** needs interoperability with Virtual Observatory tools

The main formats used in astronomy:

| Format | Extension | Binary? | Self-describing? | Human-readable? | Best for |
|--------|-----------|---------|-----------------|-----------------|----------|
| **FITS** | `.fits` | Yes | Yes (header) | No | Large catalogs, images, spectra |
| **VOTable** | `.xml`, `.vot` | Optional | Yes (XML) | Yes (XML) | VO tools, queries |
| **ECSV** | `.ecsv`, `.csv` | No | Yes (YAML header) | Yes | Small tables with metadata |
| **HDF5** | `.hdf5`, `.h5` | Yes | Yes (attributes) | No | Large structured datasets |
| **CSV** | `.csv`, `.txt` | No | No | Yes | Simple exchange, spreadsheets |

---

## 2. Setup and creating a working subsample

We start by loading a representative subsample from the Gaia DR2 FITS catalog and saving it in all formats for comparison.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

# astropy — the main library for astronomical data in Python
# Documentation: https://docs.astropy.org/en/stable/
import astropy
from astropy.io import fits          # FITS I/O
from astropy.io import ascii         # Text format I/O  
from astropy.io.votable import writeto as votable_write  # VOTable
from astropy.table import Table, Column, MaskedColumn  # Table object
import astropy.units as u
from astropy.coordinates import SkyCoord

import h5py  # HDF5 file access (also via astropy.table)

# Working directory for outputs
OUTDIR = 'format_examples'
os.makedirs(OUTDIR, exist_ok=True)

print(f'astropy version: {astropy.__version__}')
print(f'Output directory: {OUTDIR}/')
print('All imports successful!')

In [ ]:
# ================================================================
# Data: Gaia DR2 subsample — stars with 16 ≤ G < 17 mag
# This magnitude slice contains ~78.9 million stars (13.4 GB).
# We work with 10,000 of them for this tutorial.
# ================================================================
# ================================================================
GAIA_FILE = '/home/comparat/data/gaia_cat/table_16_g_17.fits'

print('Opening Gaia DR2 FITS file (memory-mapped)...')
with fits.open(GAIA_FILE, memmap=True) as hdul:
    N_TOTAL = hdul[1].header['NAXIS2']
    print(f'  Total stars in file: {N_TOTAL:,}')

    # Take every 7892nd star => 10,000 stars
    N_SAMPLE = 10_000
    stride = N_TOTAL // N_SAMPLE
    idx = np.arange(0, N_TOTAL, stride)[:N_SAMPLE]

    d = hdul[1].data

    # Read all columns for the subsample
    source_id = d['source_id'][idx]
    ra        = d['ra'][idx].astype(np.float64)
    ra_err    = d['ra_error'][idx].astype(np.float64)
    dec       = d['dec'][idx].astype(np.float64)
    dec_err   = d['dec_error'][idx].astype(np.float64)
    plx       = d['parallax'][idx].astype(np.float64)
    plx_err   = d['parallax_error'][idx].astype(np.float64)
    pmra      = d['pmra'][idx].astype(np.float64)
    pmra_err  = d['pmra_error'][idx].astype(np.float64)
    pmdec     = d['pmdec'][idx].astype(np.float64)
    pmdec_err = d['pmdec_error'][idx].astype(np.float64)
    G_mag     = d['phot_g_mean_mag'][idx].astype(np.float64)
    BP_mag    = d['phot_bp_mean_mag'][idx].astype(np.float64)
    RP_mag    = d['phot_rp_mean_mag'][idx].astype(np.float64)

print(f'\nSubsample loaded: {N_SAMPLE:,} stars (every {stride}th row)')

---

## 3. The FITS format

**FITS** (Flexible Image Transport System) is the dominant format in astronomy since the 1970s. It was standardised by the IAU and NASA and is maintained by the FITS Support Office at NASA GSFC.

Reference: FITS Standard 4.0 — [https://fits.gsfc.nasa.gov/fits_standard.html](https://fits.gsfc.nasa.gov/fits_standard.html)

### 3.1 Structure of a FITS file

A FITS file consists of one or more **Header Data Units (HDUs)**:

```
FITS file
├── Primary HDU (HDU[0])
│   ├── Header (ASCII, 80-char cards, metadata)
│   └── Data  (optional: image, array, or empty)
├── Extension HDU 1 (HDU[1])
│   ├── Header
│   └── Data (BinTableHDU, ImageHDU, etc.)
└── Extension HDU 2 ...
```

For catalogs, the data is stored in a **Binary Table Extension** (`BinTableHDU`), which stores typed columns (float64, int32, string, etc.) in compact binary format.

### 3.2 FITS data types

| FITS format code | Python/NumPy type | Description |
|-----------------|-------------------|-------------|
| `D` | float64 | Double precision float |
| `E` | float32 | Single precision float |
| `J` | int32 | 32-bit integer |
| `K` | int64 | 64-bit integer |
| `I` | int16 | 16-bit integer |
| `L` | bool | Logical (True/False) |
| `A` | str | Character string |

In [ ]:
# ================================================================
# 3.3 Reading a FITS file with astropy.io.fits
# ================================================================
print('=== Reading FITS file with astropy.io.fits ===')

with fits.open(GAIA_FILE, memmap=True) as hdul:
    # Inspect the file structure
    print('\nFile info:')
    hdul.info()

    # Primary HDU header
    print('\nPrimary HDU header keys:', list(hdul[0].header.keys())[:6])
    print(f'VOTMETA = {hdul[0].header.get("VOTMETA", "N/A")}  (contains VOTable metadata)')

    # Binary table header
    hdr = hdul[1].header
    print(f'\nBinary table:')
    print(f'  NAXIS1 = {hdr["NAXIS1"]} bytes/row')
    print(f'  NAXIS2 = {hdr["NAXIS2"]:,} rows (stars)')
    print(f'  TFIELDS = {hdr["TFIELDS"]} columns')
    print(f'  DATE-HDU = {hdr["DATE-HDU"]}  (file creation date)')

    # Column descriptions
    print('\nColumn definitions (name, format, unit):')
    for col in hdul[1].columns:
        print(f'  {col.name:30s}  {col.format:6s}  {col.unit or "—"}')

In [ ]:
# ================================================================
# 3.4 Writing a FITS file: two approaches
# ================================================================

# --- Approach A: Low-level astropy.io.fits ---
# This gives you full control over headers and column definitions

# Create column definitions
col_ra  = fits.Column(name='ra',  format='D', array=ra,      unit='deg')
col_dec = fits.Column(name='dec', format='D', array=dec,     unit='deg')
col_plx = fits.Column(name='parallax', format='D', array=plx, unit='mas')
col_G   = fits.Column(name='phot_g_mean_mag', format='D', array=G_mag, unit='mag')
col_BP  = fits.Column(name='phot_bp_mean_mag', format='D', array=BP_mag, unit='mag')
col_RP  = fits.Column(name='phot_rp_mean_mag', format='D', array=RP_mag, unit='mag')

# Create the binary table HDU
table_hdu = fits.BinTableHDU.from_columns(
    [col_ra, col_dec, col_plx, col_G, col_BP, col_RP]
)

# Add metadata to the header
table_hdu.header['SURVEY']  = 'Gaia DR2'
table_hdu.header['GMIN']    = (16.0, 'Minimum G magnitude')
table_hdu.header['GMAX']    = (17.0, 'Maximum G magnitude')
table_hdu.header['NSTARS']  = (N_SAMPLE, 'Number of stars in this file')
table_hdu.header['AUTHOR']  = 'Tutorial notebook'
table_hdu.header['REFPAPER'] = 'Gaia Collaboration, Brown et al. 2018, A&A 616, A1'
table_hdu.header['REFARVX'] = 'arXiv:1804.09365'
table_hdu.header['COMMENT'] = 'Gaia DR2 subsample: 16 <= G < 17 mag'

# Combine with empty primary HDU
primary_hdu = fits.PrimaryHDU()
hdul_out = fits.HDUList([primary_hdu, table_hdu])

# Write to file
out_path_fits = f'{OUTDIR}/gaia_subsample_lowlevel.fits'
hdul_out.writeto(out_path_fits, overwrite=True)
print(f'Written: {out_path_fits}  ({os.path.getsize(out_path_fits)/1e3:.1f} kB)')

# Verify by reading back
with fits.open(out_path_fits) as f:
    print(f'  Verification: {len(f[1].data):,} rows, columns: {f[1].columns.names}')

---

## 4. The `astropy.table.Table` — the universal interface

The **astropy Table** is the recommended high-level interface for tabular data. It:
- Supports **units** on columns (`astropy.units`)
- Handles **masked values** (like NaN, but type-aware)
- Reads and writes **all major formats** through a single `Table.read()` / `Table.write()` interface
- Supports **column metadata** (description, units, format)
- Allows **vectorised operations** and numpy-compatible slicing

Documentation: [https://docs.astropy.org/en/stable/table/index.html](https://docs.astropy.org/en/stable/table/index.html)

### 4.1 Creating an astropy Table with units and metadata

In [ ]:
# ================================================================
# Build an astropy Table with full metadata
# ================================================================

# Create a Table from numpy arrays, attaching units to each column
gaia_table = Table()

gaia_table['ra']       = Column(ra,       unit=u.deg,     description='Right Ascension (ICRS, J2015.5)')
gaia_table['dec']      = Column(dec,      unit=u.deg,     description='Declination (ICRS, J2015.5)')
gaia_table['ra_error'] = Column(ra_err,   unit=u.mas,     description='RA uncertainty')
gaia_table['dec_error']= Column(dec_err,  unit=u.mas,     description='Dec uncertainty')
gaia_table['parallax'] = Column(plx,      unit=u.mas,     description='Trigonometric parallax')
gaia_table['parallax_error'] = Column(plx_err, unit=u.mas, description='Parallax uncertainty')
gaia_table['pmra']     = Column(pmra,     unit=u.mas/u.yr, description='Proper motion in RA (mu_alpha*cos(delta))')
gaia_table['pmdec']    = Column(pmdec,    unit=u.mas/u.yr, description='Proper motion in Dec')
gaia_table['G']        = Column(G_mag,    unit=u.mag,     description='Gaia G-band mean magnitude')
gaia_table['BP']       = Column(BP_mag,   unit=u.mag,     description='Gaia BP-band mean magnitude')
gaia_table['RP']       = Column(RP_mag,   unit=u.mag,     description='Gaia RP-band mean magnitude')

# Table-level metadata
gaia_table.meta['name']        = 'Gaia DR2 subsample'
gaia_table.meta['survey']      = 'Gaia Data Release 2'
gaia_table.meta['reference']   = 'Gaia Collaboration, Brown et al. 2018, A&A 616, A1'
gaia_table.meta['arxiv']       = 'arXiv:1804.09365'
gaia_table.meta['g_min']       = 16.0
gaia_table.meta['g_max']       = 17.0
gaia_table.meta['n_total']     = N_TOTAL
gaia_table.meta['n_sample']    = N_SAMPLE
gaia_table.meta['sample_stride'] = int(stride)
gaia_table.meta['ref_epoch']   = 'J2015.5'

print(f'Table created: {len(gaia_table):,} rows × {len(gaia_table.colnames)} columns')
print(f'Columns: {gaia_table.colnames}')
print()
gaia_table.pprint(max_lines=10)

### 4.2 Working with an astropy Table: filtering, sorting, adding columns

In [ ]:
# --- 4.2.1 Adding derived columns ---

# BP-RP colour index
gaia_table['BP_RP'] = gaia_table['BP'] - gaia_table['RP']
gaia_table['BP_RP'].unit = u.mag
gaia_table['BP_RP'].description = 'BP-RP colour index'

# Parallax signal-to-noise
plx_arr = np.array(gaia_table['parallax'])
err_arr = np.array(gaia_table['parallax_error'])
gaia_table['parallax_snr'] = np.where(err_arr > 0, plx_arr / err_arr, np.nan)
gaia_table['parallax_snr'].description = 'Parallax signal-to-noise ratio'

# Distance (kpc), only for reliable parallaxes (S/N > 3)
snr = np.array(gaia_table['parallax_snr'])
dist_kpc = np.where((snr > 3) & (plx_arr > 0), 1.0 / plx_arr, np.nan)
gaia_table['distance_kpc'] = dist_kpc
gaia_table['distance_kpc'].unit = u.kpc
gaia_table['distance_kpc'].description = 'Distance estimate 1/parallax (S/N>3 only)'

print('Added columns: BP_RP, parallax_snr, distance_kpc')
print(f'New column list: {gaia_table.colnames}')

In [ ]:
# --- 4.2.2 Filtering (selecting rows) ---

# Boolean mask: reliable parallax AND blue stars (BP-RP < 1.5)
mask_nearby = (
    np.isfinite(gaia_table['parallax_snr']) &
    (gaia_table['parallax_snr'] > 5) &
    np.isfinite(gaia_table['BP_RP'])
)

nearby = gaia_table[mask_nearby]
print(f'Stars with parallax S/N > 5: {len(nearby):,}')

# Selection by column value
blue_stars = gaia_table[gaia_table['BP_RP'] < 0.5]
red_stars  = gaia_table[gaia_table['BP_RP'] > 2.5]

print(f'Blue stars  (BP-RP < 0.5): {len(blue_stars):,}')
print(f'Red stars   (BP-RP > 2.5): {len(red_stars):,}')

# --- 4.2.3 Sorting ---
gaia_table_sorted = gaia_table.copy()
gaia_table_sorted.sort('parallax', reverse=True)  # Brightest parallax first
print(f'\nTop 5 closest stars (largest parallax):')
gaia_table_sorted[['ra', 'dec', 'parallax', 'parallax_error', 'G', 'BP_RP']][:5].pprint()

# --- 4.2.4 Column statistics ---
print('\nColumn statistics:')
for col in ['G', 'BP_RP', 'parallax', 'distance_kpc']:
    arr = np.array(gaia_table[col], dtype=float)
    finite = arr[np.isfinite(arr)]
    print(f'  {col:15s}: min={finite.min():.3g}  median={np.median(finite):.3g}  max={finite.max():.3g}  N_finite={len(finite):,}')

In [ ]:
# --- 4.2.5 Converting units with astropy ---

# The Gaia DR2 FITS file does not store unit metadata for most columns.
# We must assign the known units manually before converting.
ra_val  = gaia_table['ra'][0]         # plain float [deg]
pm_val  = gaia_table['pmra'][0]       # plain float [mas/yr]
d_kpc   = gaia_table['distance_kpc']  # column [kpc]

# Wrap with the correct unit to get an astropy Quantity
ra_q  = ra_val  * u.deg
pm_q  = pm_val  * (u.mas / u.yr)

print(f'RA  = {ra_q:.4f}')
print(f'    = {ra_q.to(u.rad):.6f}')
print(f'    = {ra_q.to(u.arcsec):.2f}')

print(f'pmra = {pm_q:.4f}')
print(f'     = {pm_q.to(u.arcsec/u.yr):.6f}')

# Distance in light years
mask_finite = np.isfinite(d_kpc)
d_lyr = d_kpc[mask_finite] * u.kpc.to(u.lyr)
print(f'Distance range: {d_kpc[mask_finite].min():.3f} - {d_kpc[mask_finite].max():.3f} kpc')
print(f'              = {d_lyr.min():.0f} - {d_lyr.max():.0f} lyr')


---

## 5. Writing and reading in all formats

The astropy Table interface uses a **unified I/O system**: the same `Table.write(filename, format='...')` call works for all supported formats. Similarly, `Table.read(filename, format='...')` reads any supported format.

For large tables, the format choice significantly impacts file size and read speed.

Documentation: [https://docs.astropy.org/en/stable/io/unified.html](https://docs.astropy.org/en/stable/io/unified.html)

### 5.1 FITS format (via astropy Table)

In [ ]:
# ================================================================
# 5.1 FITS — via astropy.table.Table.write
# ================================================================

# Use a smaller table (5 columns) for clarity in format comparisons
t_save = gaia_table[['ra', 'dec', 'parallax', 'G', 'BP_RP']]

# --- Write ---
out_fits = f'{OUTDIR}/gaia_5col.fits'
t0 = time.time()
t_save.write(out_fits, format='fits', overwrite=True)
t_write_fits = time.time() - t0
size_fits = os.path.getsize(out_fits)
print(f'FITS write: {t_write_fits:.2f}s  |  size: {size_fits/1e3:.1f} kB  |  {out_fits}')

# --- Read back ---
t0 = time.time()
t_back_fits = Table.read(out_fits, format='fits')
t_read_fits = time.time() - t0
print(f'FITS read : {t_read_fits:.3f}s  |  {len(t_back_fits):,} rows')

# Check metadata preservation
print(f'Metadata keys: {list(t_back_fits.meta.keys())[:5]}')
print(f'Column units: {[(c, str(t_back_fits[c].unit)) for c in t_back_fits.colnames]}')

# Inspect the written FITS structure
with fits.open(out_fits) as f:
    print(f'\nFITS structure:')
    f.info()
    print(f'  Header sample: NAXIS1={f[1].header["NAXIS1"]}, NAXIS2={f[1].header["NAXIS2"]:,}')

### 5.2 VOTable format

**VOTable** is the standard data exchange format of the **Virtual Observatory (VO)** — an international framework for sharing and accessing astronomical data (IVOA: International Virtual Observatory Alliance).

- Format: XML with embedded table data
- Contains rich metadata: column descriptions, UCDs (Unified Content Descriptors), units
- Used by tools: TOPCAT, Aladin, VizieR, SIMBAD, astroquery

A **UCD** (Unified Content Descriptor) is a standardised vocabulary for describing what a column contains, e.g.:
- `pos.eq.ra` = Right Ascension
- `pos.eq.dec` = Declination
- `phys.distance.modulus` = Distance modulus

Reference: IVOA VOTable standard — [https://www.ivoa.net/documents/VOTable/](https://www.ivoa.net/documents/VOTable/)

astropy VOTable docs: [https://docs.astropy.org/en/stable/io/votable/index.html](https://docs.astropy.org/en/stable/io/votable/index.html)

In [ ]:
# ================================================================
# 5.2 VOTable (XML format)
# ================================================================

# --- Write ---
out_vot = f'{OUTDIR}/gaia_5col.xml'
t0 = time.time()
t_save.write(out_vot, format='votable', overwrite=True)
t_write_vot = time.time() - t0
size_vot = os.path.getsize(out_vot)
print(f'VOTable write: {t_write_vot:.2f}s  |  size: {size_vot/1e6:.2f} MB  |  {out_vot}')

# --- Read back ---
t0 = time.time()
t_back_vot = Table.read(out_vot, format='votable')
t_read_vot = time.time() - t0
print(f'VOTable read : {t_read_vot:.3f}s  |  {len(t_back_vot):,} rows')

# Peek at the XML structure
print('\nFirst 25 lines of the VOTable XML:')
with open(out_vot) as f:
    for i, line in enumerate(f):
        if i >= 25:
            break
        print(f'  {line}', end='')

### 5.3 ECSV — Enhanced CSV

**ECSV** (Enhanced Character-Separated Values) is a text format designed by the astropy team to solve the main weakness of plain CSV: **no metadata**.

ECSV stores:
- Column **names, units, and data types** in a YAML header (marked with `# %ECSV`)
- Table-level **metadata** (e.g., survey name, reference)
- The data itself as **plain comma-separated text**

This makes ECSV ideal for:
- Sharing results tables in papers
- Configuration files
- Small to medium catalogs that need to be human-readable

Documentation: [https://docs.astropy.org/en/stable/io/ascii/ecsv.html](https://docs.astropy.org/en/stable/io/ascii/ecsv.html)

In [ ]:
# ================================================================
# 5.3 ECSV (Enhanced CSV)
# ================================================================

# --- Write ---
out_ecsv = f'{OUTDIR}/gaia_5col.ecsv'
t0 = time.time()
t_save.write(out_ecsv, format='ascii.ecsv', overwrite=True)
t_write_ecsv = time.time() - t0
size_ecsv = os.path.getsize(out_ecsv)
print(f'ECSV write: {t_write_ecsv:.2f}s  |  size: {size_ecsv/1e6:.2f} MB  |  {out_ecsv}')

# --- Read back ---
t0 = time.time()
t_back_ecsv = Table.read(out_ecsv, format='ascii.ecsv')
t_read_ecsv = time.time() - t0
print(f'ECSV read : {t_read_ecsv:.3f}s  |  {len(t_back_ecsv):,} rows')
print(f'Units preserved: {[(c, str(t_back_ecsv[c].unit)) for c in t_back_ecsv.colnames]}')

# Read the ECSV header to understand the format
print('\nFirst 30 lines of the ECSV file (the header):')
with open(out_ecsv) as f:
    for i, line in enumerate(f):
        if i >= 30:
            print('  ...')
            break
        print(f'  {line}', end='')

### 5.4 HDF5 format

**HDF5** (Hierarchical Data Format 5) is a binary format for storing large, complex datasets. It supports:
- **Hierarchical groups** (like a filesystem within a file)
- **Datasets** of any shape and type
- **Attributes** for metadata at any level
- **Chunking and compression** for efficient storage and access
- **Partial reads** (access subsets without reading the whole file)

HDF5 is widely used in physics and astronomy (LIGO, Euclid, DESI output files).

In Python, HDF5 is accessed via the `h5py` library or through `astropy.table.Table.write(format='hdf5')`.

> **Note:** astropy stores tables in HDF5 using the `path` argument to name the dataset within the file. Multiple tables can be stored in a single HDF5 file at different paths.

In [ ]:
# ================================================================
# 5.4 HDF5 format
# ================================================================

# --- Write via astropy Table ---
out_hdf5 = f'{OUTDIR}/gaia_5col.hdf5'
t0 = time.time()
t_save.write(out_hdf5, format='hdf5', path='gaia_stars', overwrite=True)
t_write_hdf5 = time.time() - t0
size_hdf5 = os.path.getsize(out_hdf5)
print(f'HDF5 write: {t_write_hdf5:.2f}s  |  size: {size_hdf5/1e3:.1f} kB  |  {out_hdf5}')

# --- Read back ---
t0 = time.time()
t_back_hdf5 = Table.read(out_hdf5, format='hdf5', path='gaia_stars')
t_read_hdf5 = time.time() - t0
print(f'HDF5 read : {t_read_hdf5:.3f}s  |  {len(t_back_hdf5):,} rows')

# --- Inspect the HDF5 file structure with h5py ---
print('\nHDF5 file structure (h5py):')
with h5py.File(out_hdf5, 'r') as f:
    def print_structure(name, obj):
        indent = '  ' * name.count('/')
        if isinstance(obj, h5py.Dataset):
            print(f'{indent}/{name}: shape={obj.shape}, dtype={obj.dtype}')
        elif isinstance(obj, h5py.Group):
            print(f'{indent}/{name}/  [group]')
    f.visititems(print_structure)
    print('\nTop-level attributes:')
    for k, v in f.attrs.items():
        print(f'  {k}: {v}')

# --- Write multiple tables in one HDF5 file ---
out_hdf5_multi = f'{OUTDIR}/gaia_multi.hdf5'

# All stars
t_save.write(out_hdf5_multi, format='hdf5', path='all_stars', overwrite=True)
# Only nearby stars (high parallax S/N)
nearby_table = gaia_table[mask_nearby][['ra', 'dec', 'parallax', 'parallax_snr', 'G', 'BP_RP']]
nearby_table.write(out_hdf5_multi, format='hdf5', path='nearby_stars', append=True)

print(f'\nMulti-table HDF5: {out_hdf5_multi}  ({os.path.getsize(out_hdf5_multi)/1e3:.1f} kB)')
with h5py.File(out_hdf5_multi, 'r') as f:
    print('HDF5 groups:', list(f.keys()))

### 5.5 Plain CSV and text formats

In [ ]:
# ================================================================
# 5.5 CSV and text formats
# ================================================================

# Use a very small table (100 rows) for text formats
t_small = gaia_table[['ra', 'dec', 'parallax', 'G', 'BP_RP']][:100]

# --- Plain CSV ---
out_csv = f'{OUTDIR}/gaia_100.csv'
t_small.write(out_csv, format='ascii.csv', overwrite=True)
print(f'CSV: {os.path.getsize(out_csv):,} bytes')

# --- Fixed-width (good for visual inspection) ---
out_fixed = f'{OUTDIR}/gaia_100_fixed.txt'
t_small.write(out_fixed, format='ascii.fixed_width', overwrite=True)
print(f'Fixed-width: {os.path.getsize(out_fixed):,} bytes')

# --- LaTeX table (for papers!) ---
out_latex = f'{OUTDIR}/gaia_10.tex'
t_small[:10].write(out_latex, format='ascii.latex', overwrite=True)
print(f'LaTeX: {os.path.getsize(out_latex):,} bytes')

# Show the CSV
print('\nFirst 6 lines of the CSV file:')
with open(out_csv) as f:
    for i, line in enumerate(f):
        if i >= 6: break
        print(f'  {line}', end='')

print('\n\nFirst 6 lines of the fixed-width file:')
with open(out_fixed) as f:
    for i, line in enumerate(f):
        if i >= 6: break
        print(f'  {line}', end='')

# --- Read a plain CSV with no metadata (manual unit assignment) ---
print('\n\nReading CSV (no units preserved):')
t_csv_back = Table.read(out_csv, format='ascii.csv')
print(f'  Columns: {t_csv_back.colnames}')
print(f'  Column units: {[(c, str(t_csv_back[c].unit)) for c in t_csv_back.colnames]}')
print('  (Note: units are lost in plain CSV — this is why ECSV is preferred)')

# Manually re-attach units
t_csv_back['ra'].unit = u.deg
t_csv_back['dec'].unit = u.deg
t_csv_back['parallax'].unit = u.mas
t_csv_back['G'].unit = u.mag
t_csv_back['BP_RP'].unit = u.mag
print('  Units re-attached manually.')

---

## 6. Format comparison

Now that we have written the same data in all formats, let us compare them systematically.

In [ ]:
# ================================================================
# Performance and size comparison
# ================================================================

results = [
    ('FITS',    out_fits,  size_fits,   t_write_fits,  t_read_fits),
    ('VOTable', out_vot,   size_vot,    t_write_vot,   t_read_vot),
    ('ECSV',    out_ecsv,  size_ecsv,   t_write_ecsv,  t_read_ecsv),
    ('HDF5',    out_hdf5,  size_hdf5,   t_write_hdf5,  t_read_hdf5),
]

print(f'{N_SAMPLE:,} rows × 5 columns comparison:')
print(f'{"Format":10s}  {"Size (MB)":>10s}  {"Write (s)":>10s}  {"Read (s)":>10s}  {"Bytes/row":>10s}')
print('-' * 60)
for name, path, sz, tw, tr in results:
    bpr = sz / N_SAMPLE
    print(f'{name:10s}  {sz/1e6:>10.2f}  {tw:>10.3f}  {tr:>10.3f}  {bpr:>10.1f}')

print()
print('Interpretation:')
print('  FITS    : smallest binary, fast, standard in astronomy')
print('  HDF5    : similar size to FITS, supports hierarchy and partial reads')
print('  ECSV    : human-readable + metadata, ~4x larger than binary')
print('  VOTable : XML overhead makes it large, but VO-compatible')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names  = [r[0] for r in results]
sizes  = [r[2]/1e6 for r in results]
writes = [r[3] for r in results]
reads  = [r[4] for r in results]
colors = ['steelblue', 'tomato', 'forestgreen', 'orange']

# File size
ax = axes[0]
bars = ax.bar(names, sizes, color=colors, edgecolor='k', lw=0.5)
ax.set_ylabel('File size [MB]', fontsize=12)
ax.set_title(f'File size ({N_SAMPLE:,} rows × 5 columns)', fontsize=11)
for bar, sz in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{sz:.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, max(sizes) * 1.3)

# Write time
ax = axes[1]
x = np.arange(len(names))
w = 0.35
bars_w = ax.bar(x - w/2, writes, w, label='Write', color=colors, edgecolor='k', lw=0.5)
bars_r = ax.bar(x + w/2, reads, w, label='Read', color=colors, edgecolor='k', lw=0.5, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('Time [seconds]', fontsize=12)
ax.set_title('Write and read time', fontsize=11)
ax.legend()

plt.suptitle('Format comparison (same data, different formats)', fontsize=12)
plt.tight_layout()
plt.savefig('format_comparison.pdf', bbox_inches='tight')
plt.show()

---

## 7. Advanced FITS operations

### 7.1 FITS headers in depth

A FITS header is a sequence of **80-character records** (cards). Each card has:
- **Keyword** (8 characters): the key
- **Value indicator** (`=`) and **value**: can be string, integer, float, or logical
- **Comment** (after `/`)

Special cards: `COMMENT`, `HISTORY`, `END`

In [ ]:
# ================================================================
# 7.1 FITS header manipulation
# ================================================================

# Build a rich FITS file with provenance information
primary = fits.PrimaryHDU()

# Add structured provenance metadata
hdr = primary.header
hdr['SURVEY']   = ('Gaia DR2', 'Survey name')
hdr['REFPAPER'] = ('Brown+2018', 'Reference paper')
hdr['REFARXIV'] = ('1804.09365', 'arXiv identifier')
hdr['REFJOUR']  = ('A&A 616 A1', 'Journal reference')
hdr['GMIN']     = (16.0, '[mag] Minimum G-band magnitude')
hdr['GMAX']     = (17.0, '[mag] Maximum G-band magnitude')
hdr['EPOCH']    = ('J2015.5', 'Reference epoch')
hdr['COORD']    = ('ICRS', 'Coordinate system')
hdr['N_TOTAL']  = (N_TOTAL, 'Total stars in parent catalog')
hdr['N_SAMPLE'] = (N_SAMPLE, 'Stars in this file (subsample)')
hdr['STRIDE']   = (int(stride), 'Subsampling stride')

# HISTORY and COMMENT cards
hdr.add_history('Created by file_formats_formats.ipynb tutorial')
hdr.add_history(f'Data from: {GAIA_FILE}')
hdr.add_comment('This is a demonstration file for the astropy tutorial.')

# Print the header
print('FITS primary header:')
print(repr(hdr))

# Accessing header values
print(f'\nAccessing header values:')
print(f'  hdr["SURVEY"]  = {hdr["SURVEY"]}')
print(f'  hdr["GMIN"]    = {hdr["GMIN"]}')
print(f'  hdr.comments["GMIN"] = {hdr.comments["GMIN"]}')

In [ ]:
# ================================================================
# 7.2 Multi-extension FITS: storing multiple tables in one file
# ================================================================

# Create three separate tables: all stars, blue stars, red stars
t_all  = gaia_table[['ra', 'dec', 'G', 'BP_RP', 'parallax']]
t_blue = gaia_table[np.isfinite(gaia_table['BP_RP']) & (gaia_table['BP_RP'] < 0.8)][['ra','dec','G','BP_RP','parallax']]
t_red  = gaia_table[np.isfinite(gaia_table['BP_RP']) & (gaia_table['BP_RP'] > 2.5)][['ra','dec','G','BP_RP','parallax']]

print(f'Table sizes: all={len(t_all):,}  blue={len(t_blue):,}  red={len(t_red):,}')

# Convert each Table to a FITS BinTableHDU
hdu0 = fits.PrimaryHDU()
hdu0.header['COMMENT'] = 'Multi-extension FITS with Gaia subsamples'

hdu_all  = fits.BinTableHDU(np.array(t_all),  name='ALL_STARS')
hdu_blue = fits.BinTableHDU(np.array(t_blue), name='BLUE_STARS')
hdu_red  = fits.BinTableHDU(np.array(t_red),  name='RED_STARS')

hdu_all.header ['BPRPMAX'] = (0.8, 'BP-RP < 0.8 (blue stars)')
hdu_blue.header['BPRPMAX'] = (0.8, 'BP-RP < 0.8 (blue stars)')
hdu_red.header ['BPRPMIN'] = (2.5, 'BP-RP > 2.5 (red stars)')

out_multi = f'{OUTDIR}/gaia_multi_ext.fits'
fits.HDUList([hdu0, hdu_all, hdu_blue, hdu_red]).writeto(out_multi, overwrite=True)

print(f'\nMulti-extension FITS written: {out_multi}')
with fits.open(out_multi) as f:
    f.info()
    print(f'  HDU[1] {f[1].name}: {len(f[1].data):,} rows')
    print(f'  HDU[2] {f[2].name}: {len(f[2].data):,} rows')
    print(f'  HDU[3] {f[3].name}: {len(f[3].data):,} rows')

# Read a specific extension
blue_back = Table(fits.getdata(out_multi, extname='BLUE_STARS'))
print(f'\nRead BLUE_STARS extension: {len(blue_back):,} rows')

---

## 8. Data preparation best practices

### 8.1 Handling missing values (NaN and masked arrays)

In [ ]:
# ================================================================
# 8.1 Missing values
# ================================================================

# In Gaia, some stars have no BP or RP measurement (NaN in our arrays)
n_nan_bp = np.sum(np.isnan(BP_mag))
n_nan_rp = np.sum(np.isnan(RP_mag))
n_nan_plx = np.sum(np.isnan(plx))

print('Missing values in the subsample:')
print(f'  BP missing: {n_nan_bp:,} ({n_nan_bp/N_SAMPLE*100:.1f}%)')
print(f'  RP missing: {n_nan_rp:,} ({n_nan_rp/N_SAMPLE*100:.1f}%)')
print(f'  Parallax missing: {n_nan_plx:,} ({n_nan_plx/N_SAMPLE*100:.1f}%)')

# Option 1: astropy MaskedColumn (recommended for FITS/VOTable)
# Creates an array with a boolean mask (True = missing)
masked_table = Table()
masked_table['ra']  = MaskedColumn(ra,  mask=np.zeros(N_SAMPLE, dtype=bool))
masked_table['BP']  = MaskedColumn(BP_mag, mask=np.isnan(BP_mag))
masked_table['RP']  = MaskedColumn(RP_mag, mask=np.isnan(RP_mag))
masked_table['plx'] = MaskedColumn(plx,    mask=np.isnan(plx))

print(f'\nMasked table: {len(masked_table)} rows')
print(f'  Number masked in BP : {masked_table["BP"].mask.sum():,}')
print(f'  Sum ignoring missing: {masked_table["BP"].sum():.1f}  (masked values excluded)')

# Save masked table to FITS (masks stored as IEEE NaN or fill values)
out_masked = f'{OUTDIR}/gaia_masked.fits'
masked_table.write(out_masked, format='fits', overwrite=True)
print(f'\nMasked table saved: {out_masked}')

# Option 2: use fill_value when no masked column is needed
BP_filled = np.where(np.isnan(BP_mag), -9999.0, BP_mag)
print(f'\nFill value approach: {(BP_filled == -9999.0).sum()} values set to -9999')

In [ ]:
# ================================================================
# 8.2 Quality cuts and data cleaning
# ================================================================

print('Quality cuts pipeline:')
t = gaia_table.copy()
print(f'  Start: {len(t):,} stars')

# 1. Remove stars with missing photometry
t = t[np.isfinite(t['BP']) & np.isfinite(t['RP'])]
print(f'  After BP,RP not NaN: {len(t):,}')

# 2. Remove stars with BP-RP < -0.5 (unphysical) or > 6 (extreme)
t = t[(t['BP_RP'] >= -0.5) & (t['BP_RP'] <= 6.0)]
print(f'  After -0.5 < BP-RP < 6: {len(t):,}')

# 3. Keep only stars with measured parallax
t = t[np.isfinite(t['parallax'])]
print(f'  After parallax not NaN: {len(t):,}')

# 4. Keep only stars in a declination range (e.g., observable from Europe)
t = t[t['dec'] > -30.0]
print(f'  After Dec > -30°: {len(t):,}')

# Save cleaned catalog
out_clean = f'{OUTDIR}/gaia_clean.fits'
t.write(out_clean, format='fits', overwrite=True)
print(f'\nCleaned catalog saved: {out_clean}  ({os.path.getsize(out_clean)/1e3:.0f} kB)')
print(f'Columns: {t.colnames}')

In [ ]:
# ================================================================
# 8.3 Cross-matching between tables
# ================================================================
# Scenario: we have two catalogs and want to find stars in common

# Create two 'catalogs' from our sample (simulating real cross-matching)
cat_A = gaia_table[:500]
cat_B = gaia_table[450:]  # Overlap with cat_A for stars 450-499

# Build SkyCoord objects
sc_A = SkyCoord(ra=cat_A['ra'].quantity, dec=cat_A['dec'].quantity)
sc_B = SkyCoord(ra=cat_B['ra'].quantity, dec=cat_B['dec'].quantity)

# Find nearest neighbour in B for each star in A
idx_match, sep2d, _ = sc_A.match_to_catalog_sky(sc_B)

# Accept matches within 0.5 arcsec
MAX_SEP_ARCSEC = 0.5
good_match = sep2d.arcsec < MAX_SEP_ARCSEC

print(f'Cross-match: cat_A ({len(cat_A)} stars) vs. cat_B ({len(cat_B)} stars)')
print(f'  Matches within {MAX_SEP_ARCSEC}"    : {good_match.sum():,}')
print(f'  Median separation (all) : {sep2d.arcsec.mean():.4f}"')
print(f'  Median separation (matched): {sep2d.arcsec[good_match].mean():.4f}"')

# Build matched table
matched = Table()
matched['ra_A']  = cat_A['ra'][good_match]
matched['dec_A'] = cat_A['dec'][good_match]
matched['G_A']   = cat_A['G'][good_match]
matched['G_B']   = cat_B['G'][idx_match[good_match]]
matched['sep_arcsec'] = sep2d.arcsec[good_match]

print(f'\nMatched table: {len(matched)} rows')
matched[:5].pprint()

---

## 9. Summary: format cheat sheet

### Quick reference: astropy I/O

```python
from astropy.table import Table

# ---- READING ----
t = Table.read('file.fits')                    # FITS
t = Table.read('file.fits', hdu=2)             # FITS, specific HDU
t = Table.read('file.xml',  format='votable')  # VOTable
t = Table.read('file.ecsv', format='ascii.ecsv')  # ECSV
t = Table.read('file.hdf5', format='hdf5', path='mydata')  # HDF5
t = Table.read('file.csv',  format='ascii.csv')  # CSV

# ---- WRITING ----
t.write('file.fits',          overwrite=True)                 # FITS
t.write('file.xml',           format='votable',  overwrite=True)  # VOTable
t.write('file.ecsv',          format='ascii.ecsv', overwrite=True)  # ECSV
t.write('file.hdf5',          format='hdf5', path='data', overwrite=True)  # HDF5
t.write('file.csv',           format='ascii.csv',  overwrite=True)  # CSV
t.write('table.tex',          format='ascii.latex',overwrite=True)  # LaTeX

# ---- LOW-LEVEL FITS ----
from astropy.io import fits
with fits.open('file.fits') as hdul:
    hdul.info()              # List all HDUs
    data = hdul[1].data      # Data from HDU 1 (BinTable)
    header = hdul[1].header  # Header from HDU 1
    val = header['KEYWORD']  # Read header keyword
```

### Format selection guide

| Use case | Recommended format | Why |
|----------|-------------------|-----|
| Large catalogs (>1M rows) | **FITS** or **HDF5** | Binary, compact, fast |
| Results/paper tables (<10k rows) | **ECSV** | Human-readable + units |
| VO tool exchange | **VOTable** | TOPCAT, Aladin, VizieR |
| Spreadsheets / Excel | **CSV** | Widely supported |
| Multiple related datasets | **HDF5** | One file, multiple datasets |
| Long-term archiving | **FITS** | 40+ year standard in astronomy |

### Further reading

- **astropy Table documentation:** [https://docs.astropy.org/en/stable/table/index.html](https://docs.astropy.org/en/stable/table/index.html)
- **astropy unified I/O:** [https://docs.astropy.org/en/stable/io/unified.html](https://docs.astropy.org/en/stable/io/unified.html)
- **astropy FITS guide:** [https://docs.astropy.org/en/stable/io/fits/index.html](https://docs.astropy.org/en/stable/io/fits/index.html)
- **FITS standard:** [https://fits.gsfc.nasa.gov/fits_standard.html](https://fits.gsfc.nasa.gov/fits_standard.html)
- **ECSV format spec:** [https://docs.astropy.org/en/stable/io/ascii/ecsv.html](https://docs.astropy.org/en/stable/io/ascii/ecsv.html)
- **IVOA VOTable standard:** [https://www.ivoa.net/documents/VOTable/](https://www.ivoa.net/documents/VOTable/)
- **Gaia DR2 paper:** Gaia Collaboration, Brown et al. (2018), A&A 616, A1 — [arXiv:1804.09365](https://arxiv.org/abs/1804.09365)

### Exercises

1. **Format sizes:** Load 1000 stars from the Gaia file and save them in all 5 formats. Record the file sizes. Which is smallest? By what factor?

2. **Metadata preservation:** Write a table to ECSV with custom metadata (your name, date, description). Read it back and verify all metadata is preserved. Repeat with CSV — what is lost?

3. **Large file access:** Use `fits.open(file, memmap=True)` to load only rows 1,000,000–1,001,000 from the full Gaia file. How long does this take? Compare to loading from a smaller FITS file.

4. **Multi-extension FITS:** Create a FITS file with three extensions: (1) all stars, (2) stars with $|b| > 60°$, (3) stars with $|b| < 10°$. Read only extension 2.

5. **Column units:** Load the ECSV file back and convert the parallax column from mas to arcseconds using `astropy.units`. Then compute and add a distance column in parsecs.

In [ ]:
# Summary of all files created
print('Files created in this tutorial:')
print(f'{"File":50s}  {"Size":>12s}')
print('-' * 65)
for root, dirs, files in os.walk(OUTDIR):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        sz = os.path.getsize(fpath)
        print(f'{fpath:50s}  {sz/1e3:>10.1f} kB')
print()
print('Tip: delete these files when done:')
print(f'  import shutil; shutil.rmtree("{OUTDIR}")')